# ColPali smoke test (Colab)

Thin wrapper around `scripts/smoke_test.py`. All logic lives in the repository; this notebook only sets up the environment and runs the script.

**Before running:** `Runtime → Change runtime type → GPU` (L4 preferred: native bf16, as in the authors' README examples; T4 falls back to fp16).

What it checks: the checkpoint loads, a few ViDoRe pages and queries are encoded, each query ranks its own page first, and timing + peak GPU memory are recorded. This is not an evaluation run.

In [ ]:
!nvidia-smi

In [ ]:
# Idempotent: clones on first run, pulls the latest commit when the runtime is reused.
import os
if not os.path.isdir("/content/colpali-hybrid-retrieval"):
    !git clone https://github.com/ismailucrn/colpali-hybrid-retrieval.git /content/colpali-hybrid-retrieval
%cd /content/colpali-hybrid-retrieval
!git pull -q
!git log -1 --format='%H %s'

In [ ]:
# Colab preinstalls torchao 0.10, which recent peft rejects at import time ("only versions above 0.16.0
# are supported"). We do not use torchao, so remove it instead of upgrading it against Colab's torch.
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt

Optional: a Hugging Face token. The smoke-test checkpoint and dataset are not gated, so this is only needed if downloads are rate-limited. Add it under the 🔑 *Secrets* panel as `HF_TOKEN`; it is never written to the repository.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception:
    print("No HF_TOKEN secret found; continuing without it")

In [ ]:
!python scripts/smoke_test.py --config configs/smoke_test.yaml

Save the exact package versions of this environment, then print the outputs so they can be committed to the repository.


In [ ]:
!mkdir -p results/smoke_test
!pip freeze > results/smoke_test/pip_freeze_colab.txt
!ls -la results/smoke_test

In [ ]:
# Print the outputs so they can be copied into the repository (no browser download needed).
import glob
for path in sorted(glob.glob("results/smoke_test/*")):
    print(f"===== BEGIN {path} =====")
    print(open(path).read())
    print(f"===== END {path} =====")